# Project Aegis — Pill Image Classifier Training

This notebook downloads pill image data, trains a MobileNetV2 classifier, and exports it to TensorFlow.js for use in the scanner.

**Steps:**
1. Download pill images from DailyMed API + ePillID + C3PI
2. Preprocess & augment
3. Train MobileNetV2 (transfer learning)
4. Export to TF.js
5. Download the model files

**Runtime:** Select **GPU** runtime: Runtime → Change runtime type → T4 GPU

## 1. Setup & Install Dependencies

In [ ]:
!pip install -q tensorflowjs Pillow
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 2. Download Pill Images from DailyMed API

DailyMed (NLM) is the official source for FDA drug label information including pill images. We use its free API to download images for 55+ common drugs.

In [ ]:
import os
import json
import hashlib
import urllib.request
import urllib.error
import shutil
import time
from pathlib import Path

# DailyMed API (active, maintained by NLM)
DAILYMED_SPL_API = "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json"
DAILYMED_MEDIA_API = "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{setid}/media.json"

# 55+ common drugs to download images for
DRUG_LIST = [
    "acetaminophen", "ibuprofen", "aspirin", "naproxen", "amoxicillin",
    "metformin", "lisinopril", "atorvastatin", "omeprazole", "amlodipine",
    "metoprolol", "losartan", "warfarin", "gabapentin", "sertraline",
    "fluoxetine", "escitalopram", "alprazolam", "levothyroxine",
    "hydrochlorothiazide", "furosemide", "prednisone", "simvastatin",
    "clopidogrel", "tramadol", "azithromycin", "ciprofloxacin",
    "montelukast", "duloxetine", "bupropion", "aripiprazole", "quetiapine",
    "lamotrigine", "oxycodone", "celecoxib", "lorazepam", "diazepam",
    "zolpidem", "trazodone", "pantoprazole", "rosuvastatin", "famotidine",
    "ondansetron", "donepezil", "sildenafil", "rivaroxaban", "apixaban",
    "diltiazem", "carvedilol", "spironolactone", "doxycycline",
    "methylphenidate", "topiramate", "pregabalin", "meloxicam",
]

print(f"Will download images for {len(DRUG_LIST)} drugs using DailyMed API")

In [ ]:
DATA_DIR = "/content/pill_data"
RAW_DIR = os.path.join(DATA_DIR, "raw")
os.makedirs(RAW_DIR, exist_ok=True)

# Only skip chemical structure diagrams — these are abstract molecular bond
# drawings that look the same across drugs and confuse the classifier.
# We KEEP: pill photos, packaging, boxes, bottles, drug facts labels
# (all are valid ways a user might photograph their medication)
SKIP_PATTERNS = [
    "struct", "structure", "formula", "chemical", "molecular",
    "mechanism", "pathway", "metabolism", "pharmacokinetic",
]

def is_useful_image(filename):
    """Skip only chemical structure diagrams. Keep everything else (pills, boxes, labels)."""
    name_lower = filename.lower()
    for pattern in SKIP_PATTERNS:
        if pattern in name_lower:
            return False
    return True

def fetch_json(url, timeout=15):
    """Fetch JSON from a URL with error handling."""
    req = urllib.request.Request(url, headers={"User-Agent": "ProjectAegis/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode())

def download_file(url, dest_path, timeout=20):
    """Download a file, return True on success."""
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "ProjectAegis/1.0"})
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            with open(dest_path, "wb") as f:
                shutil.copyfileobj(resp, f)
        # Verify it's a real image (>1KB)
        if os.path.getsize(dest_path) > 1024:
            return True
        os.remove(dest_path)
    except Exception:
        if os.path.exists(dest_path):
            os.remove(dest_path)
    return False

def download_dailymed_images(drug_name, max_spls=10):
    """
    Download drug images from DailyMed for a given drug.
    Keeps pills, packaging, labels — only skips chemical structure diagrams.
    """
    drug_dir = os.path.join(RAW_DIR, drug_name)
    os.makedirs(drug_dir, exist_ok=True)

    count = 0
    skipped = 0
    try:
        search_url = f"{DAILYMED_SPL_API}?drug_name={urllib.request.quote(drug_name)}&pagesize={max_spls}"
        data = fetch_json(search_url)

        spl_results = data.get("data", [])
        for spl in spl_results:
            setid = spl.get("setid")
            if not setid:
                continue

            try:
                media_url = DAILYMED_MEDIA_API.format(setid=setid)
                media_data = fetch_json(media_url)

                media_obj = media_data.get("data", {})
                media_list = media_obj.get("media", []) if isinstance(media_obj, dict) else []

                for media_item in media_list:
                    img_url = media_item.get("url", "")
                    mime = media_item.get("mime_type", "")
                    name = media_item.get("name", "")

                    if not (mime.startswith("image/") or img_url.lower().endswith((".jpg", ".jpeg", ".png", ".gif"))):
                        continue

                    # Only skip chemical structure diagrams
                    if not is_useful_image(name):
                        skipped += 1
                        continue

                    url_hash = hashlib.md5(img_url.encode()).hexdigest()[:10]
                    ext = ".jpg"
                    if "png" in img_url.lower() or "png" in mime:
                        ext = ".png"
                    filename = f"{drug_name}_{setid[:8]}_{url_hash}{ext}"
                    filepath = os.path.join(drug_dir, filename)

                    if os.path.exists(filepath):
                        count += 1
                        continue

                    if download_file(img_url, filepath):
                        count += 1

                time.sleep(0.1)
            except Exception as e:
                print(f"  [warn] media fetch failed for {drug_name} SPL {setid[:8]}: {e}")

    except Exception as e:
        print(f"  [warn] SPL search failed for {drug_name}: {e}")

    return count, skipped


# Download images for all drugs
print("Downloading drug images from DailyMed...")
print("(Keeping pills, packaging, labels — only skipping chemical structure diagrams)\n")

total = 0
total_skipped = 0
results = {}
for i, drug_name in enumerate(DRUG_LIST):
    n, s = download_dailymed_images(drug_name, max_spls=15)
    results[drug_name] = n
    total += n
    total_skipped += s
    if (i + 1) % 10 == 0:
        print(f"  Progress: {i+1}/{len(DRUG_LIST)} drugs, {total} images kept, {total_skipped} structures skipped")
    time.sleep(0.2)

print(f"\n=== DailyMed Download Complete ===")
print(f"Images kept: {total} (pills + packaging + labels)")
print(f"Structures skipped: {total_skipped}")
print(f"Drugs with images: {sum(1 for v in results.values() if v > 0)}")
print(f"\nPer drug:")
for drug, n in sorted(results.items(), key=lambda x: -x[1]):
    if n > 0:
        print(f"  {drug}: {n} images")

In [ ]:
# === Clean up already-downloaded images ===
# Only removes chemical structure diagrams (abstract molecular bond drawings).
# Keeps everything useful: pills, packaging, boxes, labels, bottles.

from PIL import Image
import numpy as np

def looks_like_chemical_structure(filepath):
    """
    Detect chemical structure diagrams using image analysis.
    These are typically black lines/bonds on a white background with very
    little color — visually distinct from any real product photo.
    """
    try:
        img = Image.open(filepath).convert("RGB")
        img_small = img.resize((100, 100))
        arr = np.array(img_small, dtype=np.float32)

        # Chemical structures: >80% near-white pixels AND very low color saturation
        # (This combo avoids filtering out white pill photos or white packaging)
        white_pixels = np.mean(arr > 230, axis=(0, 1))
        white_ratio = np.mean(white_pixels)

        r, g, b = arr[:,:,0], arr[:,:,1], arr[:,:,2]
        max_rgb = np.maximum(np.maximum(r, g), b)
        min_rgb = np.minimum(np.minimum(r, g), b)
        saturation = np.mean(max_rgb - min_rgb)

        # Must be BOTH very white AND nearly zero color to be a structure diagram
        if white_ratio > 0.80 and saturation < 10:
            return True, "chemical structure (white bg + no color)"

        return False, "looks like a real photo"
    except Exception:
        return False, "could not analyze"

# Clean existing downloads
cleaned = 0
kept = 0
checked = 0

for drug_name in sorted(os.listdir(RAW_DIR)):
    drug_dir = os.path.join(RAW_DIR, drug_name)
    if not os.path.isdir(drug_dir):
        continue

    for fname in os.listdir(drug_dir):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        checked += 1
        fpath = os.path.join(drug_dir, fname)

        # Check 1: Filename says it's a structure/formula
        if not is_useful_image(fname):
            os.remove(fpath)
            cleaned += 1
            continue

        # Check 2: Image content — only flag if clearly a chemical structure
        is_struct, reason = looks_like_chemical_structure(fpath)
        if is_struct:
            os.remove(fpath)
            cleaned += 1
        else:
            kept += 1

print(f"=== Cleanup Complete ===")
print(f"Checked: {checked} images")
print(f"Removed: {cleaned} chemical structure diagrams")
print(f"Kept: {kept} useful images (pills, packaging, labels, etc.)")

# Show what's left per drug
print(f"\nImages per drug:")
for drug_name in sorted(os.listdir(RAW_DIR)):
    drug_dir = os.path.join(RAW_DIR, drug_name)
    if os.path.isdir(drug_dir):
        n = len([f for f in os.listdir(drug_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
        if n > 0:
            print(f"  {drug_name}: {n}")

## 3. Add ePillID + C3PI Datasets (Large-Scale)

This cell downloads the **NIH C3PI dataset** (~4,000 reference pill images) and **ePillID benchmark** data, then maps everything into our training folder structure. This massively increases training data.

In [ ]:
# === Download ePillID benchmark + NIH C3PI reference images ===
# This adds THOUSANDS more pill images for much better accuracy.
# C3PI alone has ~4,000 reference images; ePillID has metadata for ~13K images.

import glob
import csv
import re

# ---------- Step A: Clone ePillID benchmark ----------
print("=" * 60)
print("Step A: Cloning ePillID benchmark repo...")
print("=" * 60)
if not os.path.exists("/content/ePillID"):
    !git clone --depth 1 https://github.com/usuyama/ePillID-benchmark.git /content/ePillID
else:
    print("  Already cloned")

# Explore what's in the repo
for ext in ["*.csv", "*.tsv", "*.json", "*.txt", "*.jpg", "*.png"]:
    found = glob.glob(f"/content/ePillID/**/{ext}", recursive=True)
    if found:
        print(f"  Found {len(found)} {ext} files")

# ---------- Step B: Download NIH C3PI Reference Dataset ----------
print("\n" + "=" * 60)
print("Step B: Downloading NIH C3PI Reference pill images...")
print("=" * 60)
print("Source: NIH National Library of Medicine (data.lhncbc.nlm.nih.gov)")

C3PI_DIR = "/content/c3pi_data"
C3PI_IMAGES = os.path.join(C3PI_DIR, "images")

if not os.path.exists(C3PI_IMAGES) or len(os.listdir(C3PI_IMAGES)) < 100:
    os.makedirs(C3PI_DIR, exist_ok=True)

    # Download C3PI reference image archive
    # These are high-quality studio reference pill images from NIH
    print("  Downloading C3PI_Reference archive (this may take a few minutes)...")
    !wget -q --show-progress "https://data.lhncbc.nlm.nih.gov/public/Pills/PillProjectDisc/C3PI_Reference_iro_V3.zip" \
        -O /content/c3pi_ref.zip 2>&1 || \
     wget -q --show-progress "https://data.lhncbc.nlm.nih.gov/public/Pills/C3PI_Reference.zip" \
        -O /content/c3pi_ref.zip 2>&1 || \
     echo "Primary archive not available, trying alternate..."

    if os.path.exists("/content/c3pi_ref.zip") and os.path.getsize("/content/c3pi_ref.zip") > 10000:
        print("  Extracting...")
        !unzip -q -o /content/c3pi_ref.zip -d {C3PI_DIR}
        !rm -f /content/c3pi_ref.zip
        # Find where images ended up
        all_imgs = glob.glob(f"{C3PI_DIR}/**/*.jpg", recursive=True) + \
                   glob.glob(f"{C3PI_DIR}/**/*.JPG", recursive=True) + \
                   glob.glob(f"{C3PI_DIR}/**/*.png", recursive=True)
        print(f"  Extracted {len(all_imgs)} images from C3PI")
    else:
        print("  C3PI zip download failed — will use alternate download method")
        # Fallback: try the CSV index + individual image fetch
        !wget -q "https://data.lhncbc.nlm.nih.gov/public/Pills/PillProjectDisc/C3PI_Reference_iro_V3.csv" \
            -O {C3PI_DIR}/c3pi_index.csv 2>/dev/null || echo "  CSV index also not available"
else:
    all_imgs = glob.glob(f"{C3PI_DIR}/**/*.jpg", recursive=True) + \
               glob.glob(f"{C3PI_DIR}/**/*.JPG", recursive=True) + \
               glob.glob(f"{C3PI_DIR}/**/*.png", recursive=True)
    print(f"  Already downloaded: {len(all_imgs)} images")

# ---------- Step C: Download ePillID images from their source ----------
print("\n" + "=" * 60)
print("Step C: Processing ePillID benchmark data...")
print("=" * 60)

# The ePillID benchmark contains CSV/TSV files mapping pill types to NIH image IDs.
# Parse these to understand what images are available.
epillid_dir = "/content/ePillID"
epillid_metadata = {}

# Find all data files in ePillID
for data_file in glob.glob(f"{epillid_dir}/**/*.csv", recursive=True) + \
                 glob.glob(f"{epillid_dir}/**/*.tsv", recursive=True):
    try:
        sep = "\t" if data_file.endswith(".tsv") else ","
        with open(data_file, "r", encoding="utf-8", errors="ignore") as f:
            reader = csv.DictReader(f, delimiter=sep)
            cols = reader.fieldnames or []
            print(f"  File: {os.path.basename(data_file)}")
            print(f"    Columns: {cols}")
            row_count = 0
            for row in reader:
                row_count += 1
                # Store metadata for mapping
                for col in cols:
                    if "name" in col.lower() or "label" in col.lower():
                        name = row.get(col, "").strip().lower()
                        if name:
                            epillid_metadata[name] = row
                            break
            print(f"    Rows: {row_count}")
    except Exception as e:
        print(f"  Error reading {data_file}: {e}")

print(f"\n  Total unique drug labels in ePillID: {len(epillid_metadata)}")

# ---------- Step D: Map C3PI images to drug name folders ----------
print("\n" + "=" * 60)
print("Step D: Mapping C3PI images to drug name folders...")
print("=" * 60)

# Build a flexible drug name matcher
drug_name_set = {d.lower(): d for d in DRUG_LIST}

def match_drug_name(text):
    """Match a text string to one of our known drug names."""
    text_lower = text.lower().strip()
    for known, canonical in drug_name_set.items():
        if known in text_lower:
            return canonical
    return None

# C3PI images often have NDC codes or drug names in their path/filename
# Try to map them using directory names and filenames
c3pi_mapped = 0
all_c3pi_images = glob.glob(f"{C3PI_DIR}/**/*.jpg", recursive=True) + \
                  glob.glob(f"{C3PI_DIR}/**/*.JPG", recursive=True) + \
                  glob.glob(f"{C3PI_DIR}/**/*.png", recursive=True)

for img_path in all_c3pi_images:
    # Check parent directory name and filename for drug name matches
    dirname = os.path.basename(os.path.dirname(img_path)).lower()
    filename = os.path.basename(img_path).lower()
    search_text = f"{dirname} {filename}"

    matched = match_drug_name(search_text)
    if matched:
        dest_dir = os.path.join(RAW_DIR, matched)
        os.makedirs(dest_dir, exist_ok=True)
        dest = os.path.join(dest_dir, f"c3pi_{os.path.basename(img_path)}")
        if not os.path.exists(dest):
            shutil.copy2(img_path, dest)
            c3pi_mapped += 1

print(f"  Mapped {c3pi_mapped} C3PI images to drug folders by name matching")

# Also try mapping via ePillID metadata (if it has image paths or NDC->name mappings)
epillid_mapped = 0
for label_name, row in epillid_metadata.items():
    matched = match_drug_name(label_name)
    if not matched:
        continue

    # Check if row has image path references
    for col_name, value in row.items():
        if not value:
            continue
        col_lower = col_name.lower()
        if any(k in col_lower for k in ["image", "file", "path", "photo"]):
            # Try to find this image in the ePillID directory
            candidates = [
                os.path.join(epillid_dir, value.strip()),
                os.path.join(epillid_dir, "images", value.strip()),
                os.path.join(epillid_dir, "data", value.strip()),
            ]
            for cand in candidates:
                if os.path.exists(cand) and os.path.isfile(cand):
                    dest_dir = os.path.join(RAW_DIR, matched)
                    os.makedirs(dest_dir, exist_ok=True)
                    dest = os.path.join(dest_dir, f"epillid_{os.path.basename(cand)}")
                    if not os.path.exists(dest):
                        shutil.copy2(cand, dest)
                        epillid_mapped += 1
                    break

print(f"  Mapped {epillid_mapped} images from ePillID metadata")

# ---------- Step E: Download additional images from DailyMed for sparse classes ----------
print("\n" + "=" * 60)
print("Step E: Filling sparse classes with additional DailyMed images...")
print("=" * 60)

extra_dm = 0
for drug_name in DRUG_LIST:
    drug_dir = os.path.join(RAW_DIR, drug_name)
    os.makedirs(drug_dir, exist_ok=True)
    existing = len([f for f in os.listdir(drug_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    if existing >= 15:
        continue  # Already have enough

    # Fetch more SPLs for this drug
    n = download_dailymed_images(drug_name, max_spls=20)
    extra_dm += n
    time.sleep(0.2)

print(f"  Downloaded {extra_dm} additional images from DailyMed")

# ---------- Summary ----------
print("\n" + "=" * 60)
print("=== FULL DATASET SUMMARY ===")
print("=" * 60)
total_imgs = 0
drug_counts = {}
for drug_name in sorted(os.listdir(RAW_DIR)):
    dp = os.path.join(RAW_DIR, drug_name)
    if os.path.isdir(dp):
        n = len([f for f in os.listdir(dp) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
        if n > 0:
            drug_counts[drug_name] = n
            total_imgs += n

print(f"\nDrugs with images: {len(drug_counts)}")
for drug, n in sorted(drug_counts.items(), key=lambda x: -x[1]):
    bar = "█" * min(n, 40)
    print(f"  {drug:25s} {n:4d} {bar}")

print(f"\nTOTAL: {total_imgs} images across {len(drug_counts)} drug classes")
print("\nThis data will be used by the train/val split in the next cell.")

## 4. Split into Train/Validation Sets

In [ ]:
import random

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
MIN_IMAGES = 2  # Minimum images per class to include
VAL_RATIO = 0.2

for d in [TRAIN_DIR, VAL_DIR]:
    os.makedirs(d, exist_ok=True)

total_train = 0
total_val = 0
included_classes = []

for drug_name in sorted(os.listdir(RAW_DIR)):
    drug_path = os.path.join(RAW_DIR, drug_name)
    if not os.path.isdir(drug_path):
        continue

    images = [f for f in os.listdir(drug_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    if len(images) < MIN_IMAGES:
        print(f"  Skipping {drug_name}: only {len(images)} images")
        continue

    random.seed(42)  # Reproducible split
    random.shuffle(images)

    val_count = max(1, int(len(images) * VAL_RATIO))
    val_images = images[:val_count]
    train_images = images[val_count:]

    train_drug = os.path.join(TRAIN_DIR, drug_name)
    val_drug = os.path.join(VAL_DIR, drug_name)
    os.makedirs(train_drug, exist_ok=True)
    os.makedirs(val_drug, exist_ok=True)

    for img in train_images:
        shutil.copy2(os.path.join(drug_path, img), os.path.join(train_drug, img))
    for img in val_images:
        shutil.copy2(os.path.join(drug_path, img), os.path.join(val_drug, img))

    total_train += len(train_images)
    total_val += len(val_images)
    included_classes.append(drug_name)

print(f"\n=== Data Split ===")
print(f"Classes: {len(included_classes)}")
print(f"Train images: {total_train}")
print(f"Val images: {total_val}")
print(f"\nClasses: {', '.join(included_classes)}")

## 5. Build & Train MobileNetV2 Model

Two-phase training:
1. **Phase 1**: Train only the classification head (fast convergence)
2. **Phase 2**: Fine-tune top MobileNet layers (better accuracy)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_PHASE1 = 15
EPOCHS_PHASE2 = 20

# Strong augmentation for pill images
# Pills can be rotated at any angle, different lighting, backgrounds
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=360,          # Pills can be at any angle
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.6, 1.4],  # Lighting variation
    fill_mode="nearest",
)

val_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical"
)

num_classes = len(train_gen.class_indices)
print(f"\nNumber of classes: {num_classes}")
print(f"Class labels: {list(train_gen.class_indices.keys())}")

In [ ]:
# Build model
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # Freeze for Phase 1

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(512, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
# Phase 1: Train classification head
print("=" * 50)
print("PHASE 1: Training classification head")
print("=" * 50)

callbacks_p1 = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_accuracy"),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
]

history1 = model.fit(
    train_gen,
    epochs=EPOCHS_PHASE1,
    validation_data=val_gen,
    callbacks=callbacks_p1,
)

In [ ]:
# Phase 2: Fine-tune top layers of MobileNet
print("=" * 50)
print("PHASE 2: Fine-tuning MobileNet top layers")
print("=" * 50)

base_model.trainable = True
# Freeze all but top 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # Lower LR for fine-tuning
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True, monitor="val_accuracy"),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
    tf.keras.callbacks.ModelCheckpoint(
        "/content/best_pill_model.keras",
        save_best_only=True,
        monitor="val_accuracy"
    ),
]

history2 = model.fit(
    train_gen,
    epochs=EPOCHS_PHASE2,
    validation_data=val_gen,
    callbacks=callbacks_p2,
)

In [ ]:
# Evaluate
print("\n=== Final Evaluation ===")
loss, acc = model.evaluate(val_gen)
print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {acc:.4f} ({acc*100:.1f}%)")

## 6. Plot Training History

In [ ]:
import matplotlib.pyplot as plt

# Combine histories
acc_hist = history1.history['accuracy'] + history2.history['accuracy']
val_acc_hist = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss_hist = history1.history['loss'] + history2.history['loss']
val_loss_hist = history1.history['val_loss'] + history2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(acc_hist, label='Train Acc')
ax1.plot(val_acc_hist, label='Val Acc')
ax1.axvline(x=len(history1.history['accuracy'])-0.5, color='gray', linestyle='--', label='Fine-tune start')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()

ax2.plot(loss_hist, label='Train Loss')
ax2.plot(val_loss_hist, label='Val Loss')
ax2.axvline(x=len(history1.history['loss'])-0.5, color='gray', linestyle='--', label='Fine-tune start')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

## 7. Export to TensorFlow.js

This converts the model to a format that runs in the browser via TF.js.

In [ ]:
import tensorflowjs as tfjs

OUTPUT_DIR = "/content/pill_model_output"
TFJS_DIR = os.path.join(OUTPUT_DIR, "pill-classifier")
os.makedirs(TFJS_DIR, exist_ok=True)

# Save TF.js model
tfjs.converters.save_keras_model(model, TFJS_DIR)
print(f"TF.js model saved to {TFJS_DIR}")

# Save labels.json (maps class index to drug name)
labels = {v: k for k, v in train_gen.class_indices.items()}
labels_list = [labels[i] for i in range(num_classes)]
labels_path = os.path.join(TFJS_DIR, "labels.json")
with open(labels_path, "w") as f:
    json.dump(labels_list, f, indent=2)

print(f"Labels saved: {labels_list}")
print(f"\nFiles in output:")
for f in os.listdir(TFJS_DIR):
    size = os.path.getsize(os.path.join(TFJS_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

In [ ]:
# Also save the full Keras model for future fine-tuning
model.save(os.path.join(OUTPUT_DIR, "pill_classifier_full.keras"))
print("Full Keras model saved for future fine-tuning")

## 8. Download the Model

Run this cell to download the TF.js model as a zip file. Then extract it to:
```
molecular-ai/public/models/pill-classifier/
```

In [ ]:
# Zip and download
!cd /content/pill_model_output && zip -r /content/pill-classifier-tfjs.zip pill-classifier/

from google.colab import files
files.download("/content/pill-classifier-tfjs.zip")

print("\n" + "=" * 50)
print("DONE! Extract the zip to:")
print("  molecular-ai/public/models/pill-classifier/")
print("")
print("It should contain:")
print("  - model.json")
print("  - group1-shard*.bin (weight files)")
print("  - labels.json")
print("=" * 50)

## 9. (Optional) Mount Google Drive to Save Permanently

If you want to keep the model and dataset across Colab sessions:

In [ ]:
# Uncomment to save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/pill_model_output /content/drive/MyDrive/ProjectAegis_PillModel/
# !cp -r /content/pill_data /content/drive/MyDrive/ProjectAegis_PillData/
# print("Saved to Google Drive!")